# Embedding space overlap analysis (2014)

In [54]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from typing import List
from math import log2

## 1. Read embedding files

In [55]:
def path(extension: str) -> str:
    return f'../../../Amazon_Sports_and_Outdoors/Amazon_Sports_and_Outdoors.{extension}'

dfs = {
    'noise': pd.read_csv(path('noise'), sep='\t'),
    'noise2': pd.read_csv(path('noise2'), sep='\t'),
    'image': pd.read_csv(path('image'), sep='\t'),
    'title': pd.read_csv(path('title'), sep='\t')
}

## 2. Preprocess embeddings

In [56]:
for df in dfs.values():
    df.rename(columns={"ent_id:token": "id", "ent_emb:float_seq": "emb"}, inplace=True)
    df["emb"] = df["emb"].apply(lambda x: np.fromstring(x, dtype=np.float32, sep=' '))

In [57]:
dfs['noise'].head(5)

,id,emb
0,0,"[-0.39058912, -0.54887104, -0.76889396, -0.559..."
1,1,"[2.0177474, -1.136726, 0.57648325, 0.6331659, ..."
2,3,"[-1.8498262, 0.29843494, -0.2580807, 0.6745085..."
3,2,"[0.8671691, -0.14107643, -0.44316986, 0.787506..."
4,4,"[0.0984219, -0.25746712, -0.87232226, -1.68549..."


In [58]:
dfs['noise2'].head(5)

,id,emb
0,0,"[1.0588099, -0.23163132, 0.98108613, 0.8715061..."
1,1,"[1.3922434, 0.37903383, 1.9464209, -1.6024971,..."
2,3,"[0.69762784, -0.08789875, 0.90860254, 0.610294..."
3,2,"[0.7749736, 0.27397707, -0.87506765, 1.1967739..."
4,4,"[-0.39855796, 1.1571202, 0.7237632, 0.53606665..."


In [59]:
dfs['image'].head(5)

,id,emb
0,0,"[0.014428207, -0.01609465, 0.03294648, 0.01099..."
1,1,"[0.013374746, -0.0040851366, 0.017846484, 0.00..."
2,3,"[0.025485406, 0.044892143, 0.06540884, 0.07273..."
3,2,"[0.0927549, 0.0003234906, 0.0068842433, 0.0564..."
4,4,"[0.026845569, -0.0059881625, 0.06385405, -0.01..."


In [60]:
dfs['title'].head(5)

,id,emb
0,0,"[0.059633706, 0.06158079, 0.085539125, 0.09338..."
1,1,"[0.019596875, 0.13992491, -0.016762894, 0.0414..."
2,3,"[0.037360948, 0.09006915, 0.030246628, 0.02277..."
3,2,"[0.0773414, 0.12009948, 0.01776859, -0.0058604..."
4,4,"[0.027500087, 0.0679503, 0.06407129, 0.0262953..."


## 3. Fit KNN with static embeddings

In [61]:
data = {}
knns = {}
for name, df in dfs.items():
    data[name] = np.vstack(df['emb'].values)
    knns[name] = NearestNeighbors(n_neighbors=100, metric='cosine')
    knns[name].fit(data[name])

## 4. Metrics

In [62]:
def iou_at_k(neighbors_a: List[str], neighbors_b: List[str], k: int) -> float:
    set_a = set(neighbors_a[:k])
    set_b = set(neighbors_b[:k])
    return len(set_a & set_b) / len(set_a | set_b) if set_a | set_b else 0.0

def recall_at_k(neighbors_a: List[str], neighbors_b: List[str], k: int) -> float:
    set_a = set(neighbors_a[:k])
    set_b = set(neighbors_b[:k])
    return len(set_a & set_b) / len(set_a) if set_a else 0.0

def dcg_at_k(relevance: List[int], k: int) -> float:
    return sum(rel / log2(idx + 2) for idx, rel in enumerate(relevance[:k]))

def ndcg_between(neighbors_a: List[str], neighbors_b: List[str], k: int) -> float:
    ideal_relevance = [k - i for i in range(min(k, len(neighbors_a)))]
    position_map = {cid: i for i, cid in enumerate(neighbors_a[:k])}
    relevance_b = []
    for cid in neighbors_b[:k]:
        if cid in position_map:
            relevance_b.append(k - position_map[cid])
        else:
            relevance_b.append(0)
    dcg = dcg_at_k(relevance_b, k)
    ideal_dcg = dcg_at_k(ideal_relevance, k)
    return dcg / ideal_dcg if ideal_dcg > 0 else 0.0

## 5. Analysis

In [63]:
original_ids = list(dfs['noise']['id'])
ids = range(len(original_ids))
BATCH_SIZE = 1000

def analyze_overlap(name_a, name_b):
    iou, recall, ndcg = 0.0, 0.0, 0.0

    for L in range(0, len(ids), BATCH_SIZE):
        R = min(L + BATCH_SIZE, len(ids))
        ids_slice = ids[L:R]

        distance_a, nearest_ids_a = knns[name_a].kneighbors([data[name_a][idx] for idx in ids_slice])
        distance_b, nearest_ids_b = knns[name_b].kneighbors([data[name_b][idx] for idx in ids_slice])

        for a_ids, b_ids in zip(nearest_ids_a, nearest_ids_b):
            iou += iou_at_k(a_ids, b_ids, 100)
            recall += recall_at_k(a_ids, b_ids, 100)
            ndcg += ndcg_between(a_ids, b_ids, 100)

    iou = iou * 100 / len(original_ids)
    recall = recall * 100 / len(original_ids)
    ndcg = ndcg * 100 / len(original_ids)

    print(f'Overlap between {name_a} and {name_b}:')
    print(f'  IOU@100: {iou:.2f}%')
    print(f'  Recall@100: {recall:.2f}%')
    print(f'  NDCG@100: {ndcg:.2f}%')
    print()

analyze_overlap('noise', 'noise2')
analyze_overlap('noise', 'image')
analyze_overlap('noise', 'title')
analyze_overlap('title', 'image')

Overlap between noise and noise2:
  IOU@100: 0.78%
  Recall@100: 1.54%
  NDCG@100: 8.44%

Overlap between noise and image:
  IOU@100: 0.77%
  Recall@100: 1.53%
  NDCG@100: 8.29%

Overlap between noise and title:
  IOU@100: 0.77%
  Recall@100: 1.53%
  NDCG@100: 8.38%

Overlap between title and image:
  IOU@100: 6.64%
  Recall@100: 11.54%
  NDCG@100: 21.26%

